# Exploring the LTEE reads

How long are the trimmed reads, and how does GC content vary between them? For each sample, this notebook explores the first 5,000 reads from mate 1.

Adapted from [CBIO7501 Lesson 06](https://khanlab.bio/cbio7501-f26/pages/06-jupyter.html) by Aziz Khan and the CBIO7501 workshop contributors. Course text is [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) and course code is MIT. The figure description and interpretation below use this project's reads.

In [ ]:
# Run from the project folder, like the scripts (lesson 02).
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd

samples = pd.read_csv("metadata/samples.tsv", sep="\t")
samples

In [ ]:
# One row per read: sample, read length and GC content (first 5,000 reads of read 1).
# A FASTQ file has four lines per read; the second one is the sequence.
rows = []
for _, s in samples.iterrows():
    with open(s["fastq_1"]) as fh:
        for i, line in enumerate(fh):
            if i == 4 * 5000:
                break
            if i % 4 == 1:
                seq = line.strip()
                gc = 100 * (seq.count("G") + seq.count("C")) / len(seq)
                rows.append({"sample": s["sample"], "length": len(seq), "gc": gc})
reads = pd.DataFrame(rows)
reads.groupby("sample")[["length", "gc"]].describe().round(1)

In [ ]:
#| label: fig-lengths
#| fig-cap: "Read length after trimming, first 5,000 reads of read 1 of each sample."
#| fig-alt: "Histogram of 15,000 trimmed reads spanning 25 to 150 bases. 63.2 percent of the reads are exactly 150 bases long."
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(reads["length"], bins=50, color="#2a78d6")
ax.set_xlabel("Read length (bases)")
ax.set_ylabel("Reads")
plt.show()

The 15,000 trimmed reads range from 25 to 150 bases. 9,485 (63.2%) are 150 bases long.

In [ ]:
# Share of reads shorter than 100 bases, per sample
(reads.assign(short=reads["length"] < 100)
      .groupby("sample")["short"].mean()
      .mul(100).round(1)
      .rename("% reads < 100 bases"))